# Урок 1.6. Base-модели и instruct-модели

**Модуль 1 «Введение в LLM» · Курс «LLM для Python-разработчиков»**

В уроке 1.5 мы выяснили, что короткий тег `qwen2.5:3b` на самом деле скрывает `qwen2.5:3b-instruct-q4_K_M`. Про квантизацию `q4_K_M` вы уже знаете (урок 1.3). Сегодня — про суффикс **`-instruct`**: что это за дообучение, чем «сырая» base-модель отличается от ассистента, и главное — мы вскроем **шаблон чата** и увидим, что диалог с LLM — это иллюзия, собранная из спецтокенов.

## 🎯 Цели урока

После этого урока вы сможете:

- объяснить, чем **base-модель** отличается от **instruct-модели** и как из первой делают вторую (SFT → preference tuning);
- прочитать **шаблон чата** (chat template) своей модели и собрать его руками из спецтокенов;
- пользоваться **raw-режимом** Ollama — и понимать, когда он нужен;
- предсказать поведение base-модели («автодополнение», а не «ответ») и не считать её сломанной;
- понимать, зачем base-модели существуют — это фундамент fine-tuning из модуля 6.

## План

1. Base-модель: автодополнение интернета
2. Путь от base к instruct
3. Шаблон чата: как роли становятся токенами
4. Выключаем шаблон: raw-режим
5. Настоящая base-модель в деле
6. Что выбирать в работе

⏱ **Время:** ~60–75 минут.

> 💡 Почти все ячейки урока — живые (Ollama из урока 1.5 должна работать). Для секции 5 понадобится крошечная base-модель на ~400 МБ — ячейки подскажут команду, а без неё просто вежливо пропустятся.

## 1. Base-модель: автодополнение интернета

Вспомните урок 1.1: предобучение (pretraining) — это триллионы токенов интернета и одна задача: **предсказать следующий токен**. Результат называется **base-моделью** (base / pretrained model). Это огромный «продолжатель документов»: он выучил язык, факты, стили и код — всё, что помогает угадывать следующий токен.

Чего в этом обучении *не было*: ни слова о том, что модель — «ассистент», что на вопросы надо «отвечать», а инструкции — «выполнять». Для base-модели ваш вопрос — не запрос, а **обрывок документа, который надо статистически правдоподобно продолжить**:

- вопрос, похожий на тему с форума → продолжится другими сообщениями форума;
- «Привет! Как дела?» → похоже на начало дневника или рассказа → продолжится рассказом;
- незакрытый список → продолжится пунктами списка (запомните этот эффект — он выстрелит в уроке 1.7).

Это не баг. Это ровно то, чему модель учили.

## 2. Путь от base к instruct

Чтобы «начитанный продолжатель» стал ассистентом, его **дообучают** (post-training). Конвейер выглядит так:

```text
 PRETRAINING                SFT                     PREFERENCE TUNING
 (предобучение)             (supervised             (RLHF / DPO)
                            fine-tuning)
+----------------+       +------------------+       +--------------------+
| триллионы      |       | десятки тысяч    |       | люди сравнивают    |
| токенов        |  -->  | пар «инструкция  |  -->  | варианты ответов;  |
| интернета,     |       | -> образцовый    |       | модель учится      |
| $миллионы,     |       | ответ», написан- |       | предпочитать       |
| месяцы на GPU  |       | ных людьми       |       | «хорошие»          |
+----------------+       +------------------+       +--------------------+
      base    ------------------------------------------>   instruct
```

Аналогия для бэкендера-нанимателя: base — это гений-выпускник, прочитавший всю библиотеку мира, но ни дня не работавший с клиентами. **SFT** — стажировка: ему показывают тысячи образцовых диалогов поддержки. **Preference tuning** — работа с наставником, который на каждую пару черновиков говорит, какой лучше. На выходе — сотрудник-ассистент (instruct).

Два важных следствия:

1. **Знания живут в pretraining, поведение — в дообучении.** RLHF не делает модель «умнее» и не добавляет фактов — он делает её послушнее, вежливее и безопаснее.
2. **Pretraining вам недоступен** (миллионы долларов), а вот дообучение — вполне: LoRA на бесплатном Colab, модуль 6 этого курса. Поэтому base-модели и публикуют — это сырьё.

In [ ]:
import httpx

OLLAMA_URL = "http://localhost:11434"
MODEL = "qwen2.5:3b"              # instruct — наш основной инструмент
BASE_MODEL = "qwen2.5:0.5b-base"  # крошечная base-модель для сравнения (~400 МБ)

try:
    tags = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=3).json()
    installed = [m["name"] for m in tags.get("models") or []]
    OLLAMA_AVAILABLE = True
except (httpx.ConnectError, httpx.TimeoutException):
    installed = []
    OLLAMA_AVAILABLE = False
    print("Ollama недоступна — живые ячейки будут пропущены (см. урок 1.5).")

BASE_READY = BASE_MODEL in installed
if OLLAMA_AVAILABLE:
    ok = "на месте" if MODEL in installed else f"НЕ найдена: ollama pull {MODEL}"
    base = "на месте" if BASE_READY else f"не скачана (для секции 5: ollama pull {BASE_MODEL})"
    print("Instruct-модель:", ok)
    print("Base-модель:    ", base)

## 3. Шаблон чата: как роли становятся токенами

В уроке 1.2 мы отправляли модели список `messages` с ролями `system`/`user`/`assistant`. Пора раскрыть секрет: **модель никогда не видит этот JSON.** Модель, как мы знаем из урока 1.1, видит только последовательность токенов. Значит, кто-то должен превратить список сообщений в одну строку — и этот кто-то — сервер (Ollama), а правило превращения называется **шаблон чата** (chat template).

Шаблон хранится прямо в модели — посмотрим настоящий:

In [ ]:
if OLLAMA_AVAILABLE:
    info = httpx.post(f"{OLLAMA_URL}/api/show", json={"model": MODEL}, timeout=10).json()
    print(info["template"])
else:
    print("[пропущено: Ollama недоступна]")

Выглядит устрашающе — это Go-шаблон с ветками для инструментов (tools — дойдём до них в модуле про агентов). Но если отбросить ветвления, суть проста. Семейство Qwen использует разметку **ChatML**:

```text
<|im_start|>system
Ты — лаконичный ассистент.<|im_end|>
<|im_start|>user
Какая столица у Франции?<|im_end|>
<|im_start|>assistant
```

Здесь `<|im_start|>` и `<|im_end|>` — **спецтокены**: во время дообучения (SFT) модель видела миллионы диалогов именно в такой разметке и выучила её «грамматику». Обратите внимание на последнюю строку: `<|im_start|>assistant` без закрытия — это **приглашение**: «дальше идёт реплика ассистента, продолжай». А `<|im_end|>` служит **стоп-токеном** (урок 1.2!): сгенерировав его, модель «закончила реплику», и сервер обрывает генерацию.

Проверим, что никакой магии нет: соберём эту строку руками и отправим через `/api/generate` с флагом `raw=true` — он говорит Ollama «не применяй шаблон, я всё сделал сам»:

In [ ]:
def chatml(messages: list[dict]) -> str:
    """Собирает messages в одну ChatML-строку — то, что делает сервер за нас."""
    parts = [f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages]
    return "".join(parts) + "<|im_start|>assistant\n"

def generate_raw(model: str, prompt: str, *, num_predict: int = 100,
                 temperature: float = 0.7, stop: list | None = None) -> str | None:
    """Отправляет текст модели КАК ЕСТЬ, без шаблона чата (raw-режим)."""
    if not OLLAMA_AVAILABLE:
        print("[пропущено: Ollama недоступна]")
        return None
    options = {"num_predict": num_predict, "temperature": temperature}
    if stop:
        options["stop"] = stop
    r = httpx.post(f"{OLLAMA_URL}/api/generate", json={
        "model": model, "prompt": prompt, "raw": True, "stream": False,
        "options": options,
    }, timeout=300)
    r.raise_for_status()
    return r.json()["response"]

messages = [
    {"role": "system", "content": "Ты — лаконичный ассистент."},
    {"role": "user", "content": "Какая столица у Франции?"},
]
prompt = chatml(messages)
print("Отправляем В ТОЧНОСТИ эту строку (repr, чтобы видеть \\n):")
print(repr(prompt))

answer = generate_raw(MODEL, prompt, stop=["<|im_end|>"], num_predict=60)
if answer is not None:
    print("\nОтвет модели:", answer.strip())

Работает: мы получили нормальный ответ ассистента, не используя `/api/chat` вообще. **Диалог — это иллюзия**: и `/api/chat` Ollama, и `client.chat.completions.create()` из урока 1.4, и интерфейс любого чат-бота — сахар над операцией «собери строку по шаблону и попроси продолжить».

Зачем это знать бэкендеру:

- **отладка**: если модель отвечает странно, первый вопрос — «а что за строка реально ушла в модель?»;
- **fine-tuning** (модуль 6): обучающие данные придётся размечать ровно этим шаблоном;
- **понимание инъекций** (урок 1.2): теперь видно, почему «системный промпт» — не барьер, а просто текст выше по документу.

## 4. Выключаем шаблон: raw-режим

А что если отправить модели текст **без всякой разметки**? Наш instruct останется дообученным «ассистентом по характеру», но без приглашения `<|im_start|>assistant` он не поймёт, что его о чём-то спрашивают:

In [ ]:
for raw_prompt in ["Столица Франции — ", "Какая столица у Франции?"]:
    completion = generate_raw(MODEL, raw_prompt, num_predict=50)
    if completion is not None:
        print(f">>> {raw_prompt!r}")
        print(completion.strip()[:300])
        print("-" * 60)

> **Пример вывода** (реальный прогон, у вас будет свой):
>
> ```text
> >>> 'Столица Франции — '
> 12,000 человек
> Во времена Второй мировой войны город был введен под защиту...
> ------------------------------------------------------------
> >>> 'Какая столица у Франции?'
> - Всё о мире
> Кто является первым министром иностранных дел в истории России?
> Стоит ли пить алкогольные напитки с белыми винами?
> Чему равен вес воздуха на Земле?
> ```

Смотрите: **та же самая модель**, которая секунду назад чётко отвечала «Париж», без шаблона превратилась в генератор случайных фрагментов интернета. Вопрос продолжился *другими вопросами* — как лента вопросов на форуме. Никакой поломки: исчезла разметка — исчез и «ассистент».

Но у instruct-модели дообучение всё же просвечивает даже в raw-режиме. Чтобы увидеть по-настоящему «сырое» поведение, нужна настоящая base-модель.

## 5. Настоящая base-модель в деле

Скачаем самую маленькую base-модель семейства — **`qwen2.5:0.5b-base`** (всего ~400 МБ):

```bash
ollama pull qwen2.5:0.5b-base
```

Забавный факт: это тот редкий случай, когда суффикс в теге Ollama *добавляет сырости*, а не убирает — дефолтный тег всегда instruct, а `-base` (у некоторых семейств `-text`) — сырая модель.

Одно честное замечание: сравнивать 0.5B-базу с 3B-инстрактом не совсем корректно — разница и в размере, и в дообучении. Но поведенческая пропасть видна невооружённым глазом, а чистый эксперимент с близнецами одного размера — в упражнении 3.

In [ ]:
QUESTIONS = ["Какая столица у Франции?", "Привет! Как дела?"]

if OLLAMA_AVAILABLE and BASE_READY:
    for q in QUESTIONS:
        print("ВОПРОС:", q)
        base_out = generate_raw(BASE_MODEL, q, num_predict=60)
        print(f"--- base ({BASE_MODEL}) продолжает:")
        print("   ", base_out.strip().replace("\n", " | ")[:230])
        r = httpx.post(f"{OLLAMA_URL}/api/chat", json={
            "model": MODEL,
            "messages": [{"role": "user", "content": q}],
            "options": {"num_predict": 60},
            "stream": False,
        }, timeout=300)
        print(f"--- instruct ({MODEL}) отвечает:")
        print("   ", r.json()["message"]["content"].strip().replace("\n", " | ")[:230])
        print("=" * 70)
elif OLLAMA_AVAILABLE:
    print(f"[пропущено: base-модель не скачана — ollama pull {BASE_MODEL}]")
else:
    print("[пропущено: Ollama недоступна]")

> **Пример вывода** (реальный прогон):
>
> ```text
> ВОПРОС: Какая столица у Франции?
> --- base продолжает:
>     - 512 | Главная » Статистика » Ораторы Французской критики | Тема: «Вот что у меня в голове...»
> --- instruct отвечает:
>     Столицей Франции является Париж.
> ======================================================================
> ВОПРОС: Привет! Как дела?
> --- base продолжает:
>     Я - Джейсон. Я - 14-летний мальчик. Странно, но мне кажется, что это мой первый день в школе...
> --- instruct отвечает:
>     Здравствуйте! Я в хорошем настроении и готов ответить на ваши вопросы...
> ```

База на приветствие ответила **началом чужого дневника** — потому что в интернете за фразой «Привет! Как дела?» чаще всего идёт художественный текст, а не услужливый ответ. Она не «глупая» и не «сломанная»: она идеально решает свою задачу — продолжать документы. Просто её задача — не ваша.

Именно из такого сырья SFT лепит ассистента. И когда в модуле 6 вы будете дообучать модель через LoRA, вы поймёте это дообучение до винтиков — включая тот самый шаблон ChatML, которым размечают обучающие диалоги.

## 6. Что выбирать в работе

| задача | что брать |
|---|---|
| чат-бот, ассистент, RAG, JSON-извлечение | **instruct** — всегда по умолчанию |
| fine-tuning под свой формат/домен (модуль 6) | base *или* instruct — зависит от задачи |
| автодополнение кода в IDE (fill-in-the-middle) | специальные base/coder-модели |
| draft-модель для ускорения (speculative decoding) | маленькая модель того же семейства |
| «продолжи текст в этом стиле» (литература, генерация данных) | base — она не станет ломать стиль «ассистентностью» |

Правило для тегов Ollama: **дефолтный тег = instruct**. Суффиксы `-base` / `-text` — сырые модели; не скачивайте их для чата.

И ещё один штрих к пониманию instruct-моделей: их фирменные привычки — «Конечно! Вот ответ:», маркированные списки к месту и не к месту, извинения — это не «личность», а **артефакты SFT-датасетов**: так были написаны образцовые ответы, на которых их учили.

## ⚠️ Частые ошибки

1. **«Скачал модель, а она несёт чушь»** — проверьте тег: если там `-base`/`-text`, модель работает ровно как задумано. Для чата нужен instruct (дефолтный тег).
2. **Считать, что модель сама остановится.** Instruct с правильным шаблоном заканчивает реплику EOS-токеном — но base-модель в свободном тексте его почти не генерирует и пишет до лимита `num_predict` (упражнение 1). А в raw-режиме со сломанной разметкой у некоторых семейств модель и вовсе начинает дописывать реплики за пользователя. Лимиты и стоп-токены задавайте явно.
3. **Опечатки в ручном шаблоне.** `<|imstart|>` вместо `<|im_start|>`, потерянный `\n` — ошибок не будет, просто спецтокен превратится в обычный текст, и качество тихо просядет. Сверяйтесь с `/api/show`.
4. **Чужой шаблон.** Разметка Llama (`[INST]`) для Qwen — не спецтокены, а мусор в тексте. Шаблон — часть конкретного семейства моделей.
5. **Ждать от instruct чистого «продолжения текста».** Он норовит ответить как ассистент («Конечно! Вот продолжение:»). Для стилизованной генерации берите base или хотя бы жёсткий few-shot (урок 1.7).
6. **Думать, что RLHF добавляет знания.** Знания — из pretraining; дообучение меняет поведение, формат и тон, но не начитанность.

## 🏋️ Упражнения

### Упражнение 1 — кто умеет останавливаться? 🛑

Отправьте через `/api/generate` (`raw=true`, `num_predict=300`) два запроса с одним и тем же вопросом: (а) инстракту — в корректной ChatML-разметке, (б) base-модели — обычным текстом. Сравните поля `done_reason` и `eval_count` и запустите несколько раз. Instruct стабильно ставит точку сразу после ответа; base — то упирается в лимит, то обрывается в случайном месте. Объясните разницу (подсказка: чем был EOS-токен в предобучении и чем стал после SFT?).

### Упражнение 2 — «память» чата своими руками 🧠

Соберите ChatML-строку диалога из четырёх сообщений: system, реплика пользователя с его именем, ответ ассистента, и новый вопрос «Как меня зовут?». Отправьте через raw-режим (уже со стоп-токеном) и убедитесь, что модель «помнит» имя. Сформулируйте, где на самом деле живёт память чата (подсказка: урок 1.1).

### Упражнение 3 — честная битва близнецов ⚔️

Вариант А (нужно скачать ~400 МБ): `ollama pull qwen2.5:0.5b` — instruct-близнец нашей базы, тот же размер. Задайте обоим один вопрос и сравните: теперь разница — только в дообучении.

Вариант Б (без скачивания): дайте base-модели промпт-«продолжение» («Список дел разработчика на понедельник:» + первый пункт) и промпт-вопрос. На каком из них base выглядит адекватной? Почему?

## 📝 Мини-квиз

**1. Чем отличается обучение base-модели от instruct-модели?**

<details><summary>Ответ</summary>

Base — только предобучение (предсказание следующего токена на текстах интернета). Instruct — тот же base плюс post-training: SFT (обучение на парах «инструкция → образцовый ответ») и preference tuning (RLHF/DPO — обучение на человеческих предпочтениях).

</details>

**2. Что делает шаблон чата?**

<details><summary>Ответ</summary>

Превращает список `messages` с ролями в одну строку со спецтокенами (у Qwen — ChatML: `<|im_start|>role ... <|im_end|>`). Модель не видит JSON — только эту строку. Шаблон хранится в модели и применяется сервером при каждом запросе.

</details>

**3. Почему instruct с шаблоном надёжно останавливается сразу после ответа, а base заканчивает непредсказуемо?**

<details><summary>Ответ</summary>

У instruct конец реплики выучен на SFT: закончив ответ, она ставит EOS-токен (у Qwen — `<|im_end|>`), и сервер обрывает генерацию. У base EOS встречался в предобучении лишь как *разделитель документов*, поэтому она генерирует его в случайном месте («документ мог бы закончиться здесь») — или не генерирует вовсе и упирается в `num_predict` (`done_reason: "length"`). База не знает, что «ответ закончен», — у неё нет понятия ответа.

</details>

**4. Зачем публикуют base-модели, если для чата они непригодны?**

<details><summary>Ответ</summary>

Это сырьё: фундамент для fine-tuning под свои задачи (модуль 6), для автодополнения кода (FIM), draft-моделей и задач «чистого продолжения текста», где ассистентские манеры только мешают.

</details>

**5. Роль `system` — это встроенный механизм модели или конвенция?**

<details><summary>Ответ</summary>

Конвенция: шаблон размечает системное сообщение спецтокенами, а дообучение приучает модель уделять этой части документа особое внимание. В base-модели никакой «системной роли» нет вовсе. Поэтому системный промпт и не является механизмом безопасности (урок 1.2).

</details>

## ✅ Решения упражнений

### Решение 1 — кто умеет останавливаться?

In [ ]:
if OLLAMA_AVAILABLE and BASE_READY:
    def generate_raw_full(model: str, prompt: str, num_predict: int = 300) -> dict:
        """Как generate_raw, но возвращает весь ответ API - нам нужна телеметрия."""
        r = httpx.post(f"{OLLAMA_URL}/api/generate", json={
            "model": model, "prompt": prompt, "raw": True, "stream": False,
            "options": {"num_predict": num_predict, "temperature": 0.7},
        }, timeout=300)
        r.raise_for_status()
        return r.json()

    question = "Назови один интересный факт о Луне."
    instruct = generate_raw_full(MODEL, chatml([{"role": "user", "content": question}]))
    base = generate_raw_full(BASE_MODEL, question)

    for name, d in [("instruct + ChatML   ", instruct), ("base + голый текст  ", base)]:
        print(f"{name} done_reason={d.get('done_reason')!r:9} "
              f"сгенерировано {d.get('eval_count')} токенов")
    print("\nХвост вывода base: ...", base["response"][-120:].replace("\n", " | "))
elif OLLAMA_AVAILABLE:
    print(f"[пропущено: base-модель не скачана — ollama pull {BASE_MODEL}]")
else:
    print("[пропущено: Ollama недоступна]")

Что здесь видно (и будет видно стабильнее, если запустить несколько раз):

- **Instruct** ставит точку сразу после ответа: `done_reason: "stop"` на ~100 токенах при лимите 300. На SFT её научили заканчивать реплику EOS-токеном `<|im_end|>` — сервер знает его и обрывает генерацию ровно там, где ответ закончен.
- **Base** не привязана к вашему вопросу: она либо доходит до лимита (`done_reason: "length"`), либо обрывается в **случайном месте** — тоже `"stop"`, но не потому, что «ответила»: её EOS в предобучении был всего лишь разделителем документов, и она ставит его там, где «документ мог бы закончиться». Посмотрите на хвост её вывода — он почти никогда не выглядит как завершённый ответ.

Умение замолчать *в нужном месте* — тоже приобретённый на дообучении навык. Практическое правило: с base-моделями и в raw-режиме всегда явно задавайте `num_predict` и стоп-токены.

### Решение 2 — «память» чата своими руками

In [ ]:
if OLLAMA_AVAILABLE:
    dialog = [
        {"role": "system", "content": "Ты — вежливый ассистент."},
        {"role": "user", "content": "Привет! Меня зовут Саша, я python-разработчик."},
        {"role": "assistant", "content": "Привет, Саша! Рад знакомству. Чем могу помочь?"},
        {"role": "user", "content": "Напомни, как меня зовут и кем я работаю?"},
    ]
    prompt = chatml(dialog)
    answer = generate_raw(MODEL, prompt, stop=["<|im_end|>"], num_predict=60)
    print("Ответ:", answer.strip())

Модель «вспомнила» имя и профессию — хотя мы отправили один-единственный запрос без всякой серверной сессии. **Память чата живёт в промпте**: клиент хранит историю и каждый раз отправляет её целиком, шаблон склеивает её в документ, а модель просто видит имя выше по тексту. Ровно поэтому длинный диалог рано или поздно упирается в контекстное окно (урок 1.1) — и его начинают резать или суммировать.

### Решение 3 — честная битва близнецов

In [ ]:
if OLLAMA_AVAILABLE and BASE_READY:
    twin = "qwen2.5:0.5b"   # instruct-близнец: тот же размер, но дообучен

    if twin in installed:
        # Вариант А: единственное различие - дообучение
        q = "Объясни одним предложением, что такое API."
        print("Вопрос:", q)
        print("\nbase:    ", generate_raw(BASE_MODEL, q, num_predict=60).strip()[:200])
        r = httpx.post(f"{OLLAMA_URL}/api/chat", json={
            "model": twin,
            "messages": [{"role": "user", "content": q}],
            "options": {"num_predict": 60}, "stream": False,
        }, timeout=300)
        print("instruct:", r.json()["message"]["content"].strip()[:200])
    else:
        # Вариант Б: два промпта одной base-модели
        print(f"({twin} не скачана - показываю вариант Б)\n")
        listing = generate_raw(BASE_MODEL,
                               "Список дел разработчика на понедельник:\n1. ",
                               num_predict=60, temperature=0.5)
        print("Промпт-продолжение (список):")
        print("1.", listing.strip()[:250])
        question = generate_raw(BASE_MODEL, "Что такое список дел?", num_predict=60)
        print("\nПромпт-вопрос:")
        print(question.strip()[:250])
elif OLLAMA_AVAILABLE:
    print(f"[пропущено: base-модель не скачана — ollama pull {BASE_MODEL}]")
else:
    print("[пропущено: Ollama недоступна]")

Мораль варианта Б: у base-модели «понятный» промпт — тот, что похож на документ. Начатый список она продолжает вполне осмысленно, а вопрос уводит её в случайный уголок интернета. Дать модели **образец формата и попросить продолжить** — это уже почти few-shot prompting, и именно им мы займёмся в следующем уроке.

## 🎓 Итоги

- **Base-модель** — продукт предобучения: гениальный «продолжатель документов» без понятия об «ответах» и «инструкциях». Она не сломана — у неё другая задача.
- **Instruct-модель** = base + SFT + preference tuning (RLHF/DPO). Знания — из pretraining, послушность и манеры — из дообучения.
- **Шаблон чата** превращает `messages` в одну строку со спецтокенами (`<|im_start|>…<|im_end|>` у Qwen). Модель не видит JSON; диалог — иллюзия; `/api/chat` — сахар.
- **Raw-режим** (`raw=true` в `/api/generate`) отключает шаблон: полезно для отладки, экспериментов и понимания, но требует своих стоп-токенов.
- Память чата = история в промпте. Роль `system` = конвенция шаблона и дообучения. Оба факта объясняют пределы «защиты промптом» из урока 1.2.
- Дефолтный тег в Ollama — instruct; `-base`/`-text` — сырьё для fine-tuning (модуль 6).

**Дальше — урок 1.7 «Эксперименты с промптами: few-shot и chain-of-thought»:** научимся выжимать из модели больше без всякого дообучения — примерами и рассуждениями.

## 📚 Что почитать

- [InstructGPT: Training language models to follow instructions](https://arxiv.org/abs/2203.02155) — классическая статья OpenAI про SFT + RLHF
- [Illustrating RLHF](https://huggingface.co/blog/rlhf) — разбор RLHF с картинками от Hugging Face
- [Шаблоны в Ollama](https://github.com/ollama/ollama/blob/main/docs/template.md) — как устроены и как переопределять
- [Chat Templates в transformers](https://huggingface.co/docs/transformers/chat_templating) — та же идея в экосистеме HF (пригодится в модуле 6)
- [Andrej Karpathy — State of GPT](https://www.youtube.com/watch?v=bZQun8Y4L2A) — часовой разбор конвейера base → SFT → RLHF